# 文本生成：让机器"写作"
> 难度标签：高级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：08_自然语言处理 | 源文件：文本生成.py | 核心功能：n-gram 语言模型、文本生成策略

## 概述

文本生成让模型自动生成连贯的文本。从 n-gram 语言模型到 GPT 系列，文本生成能力已经达到了惊人的水平。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

## 数学原理

### 1. 文本生成的数学框架

文本生成是自回归过程：给定前缀序列 $\mathbf{x}_{<t} = (x_1, x_2, \ldots, x_{t-1})$，预测下一个 token 的概率分布：

$$P(x_t | x_1, x_2, \ldots, x_{t-1}) = \text{softmax}(\mathbf{h}_t)$$

整个序列的联合概率由链式法则给出：

$$P(x_1, x_2, \ldots, x_T) = \prod_{t=1}^{T} P(x_t | x_{<t})$$

### 2. 字符级 RNN 模型

代码中使用字符级 LSTM 进行文本生成，模型结构：

$$\mathbf{e}_t = \text{Embedding}(x_t) \in \mathbb{R}^d$$

$$\mathbf{h}_t, \mathbf{c}_t = \text{LSTM}(\mathbf{e}_t, \mathbf{h}_{t-1}, \mathbf{c}_{t-1})$$

$$\text{logits}_t = W \mathbf{h}_t + \mathbf{b}$$

$$P(x_t = k | x_{<t}) = \frac{\exp(\text{logits}_{t,k})}{\sum_{j} \exp(\text{logits}_{t,j})}$$

### 3. 训练目标（交叉熵损失）

给定训练序列，最小化下一个字符的负对数似然：

$$\mathcal{L} = -\frac{1}{T}\sum_{t=1}^{T} \log P(x_t^* | x_{<t})$$

其中 $x_t^*$ 是真实的下一个字符。代码中 `nn.CrossEntropyLoss()` 实现此损失。

### 4. 采样策略

生成时从概率分布中采样，常用策略：

**贪心采样**：$\hat{x}_t = \arg\max_k P(x_t = k | x_{<t})$

**随机采样**：$\hat{x}_t \sim P(x_t | x_{<t})$

**温度采样**：引入温度参数 $\tau$ 控制随机性：

$$P_\tau(x_t = k) = \frac{\exp(\text{logits}_{t,k}/\tau)}{\sum_j \exp(\text{logits}_{t,j}/\tau)}$$

- $\tau \to 0$：退化为贪心采样（确定性）
- $\tau = 1$：原始分布
- $\tau \to \infty$：均匀分布（最大随机性）

**Top-k 采样**：只从概率最高的 $k$ 个 token 中采样

$$P'(x_t = k) = \begin{cases} P(x_t=k)/Z & \text{if } k \in \text{top-k} \\ 0 & \text{otherwise} \end{cases}$$

### 5. 困惑度（Perplexity）

评估生成模型质量的指标：

$$\text{PPL} = \exp\left(-\frac{1}{T}\sum_{t=1}^{T}\log P(x_t | x_{<t})\right) = \exp(\mathcal{L})$$

困惑度越低，模型对文本的预测越准确。$\text{PPL}=|V|$ 表示模型等价于随机猜测。

### 6. 隐藏状态的传递

代码中生成过程使用 `hidden` 状态传递上下文：

```python
output, hidden = model(input_seq, hidden)
```

$\mathbf{h}_t$ 编码了 $x_1, \ldots, x_t$ 的所有历史信息，是模型的"记忆"。

### 7. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `nn.Embedding(vocab_size, embed_dim)` | 嵌入矩阵 $E \in \mathbb{R}^{|V| \times d}$ |
| `nn.LSTM(embed_dim, hidden_dim)` | LSTM 单元，输出 $\mathbf{h}_t, \mathbf{c}_t$ |
| `nn.Linear(hidden_dim, vocab_size)` | 输出投影 $W \in \mathbb{R}^{|V| \times H}$ |
| `CrossEntropyLoss()` | $-\sum_k y_k \log P(x_t=k)$ |
| `torch.multinomial(probs, 1)` | 从分布 $P(x_t|x_{<t})$ 中采样 |
| `temperature` 参数 | $\tau$，控制 softmax 的锐度 |
| `hidden` 状态传递 | LSTM 的 $(\mathbf{h}_t, \mathbf{c}_t)$ 跨时间步传递 |

### 1. 简单字符级 RNN 文本生成

运行 1. 简单字符级 RNN 文本生成 的代码，观察算法在该环节的行为。

In [ ]:
print("=== 字符级 RNN 文本生成 ===")

# 训练文本
text = """人工智能是计算机科学的一个分支，它企图了解智能的实质。
机器学习是人工智能的核心，通过算法让计算机从数据中学习规律。
深度学习利用神经网络模拟人脑的工作方式。
自然语言处理让计算机理解和生成人类语言。"""

# 构建字符到索引的映射
chars = sorted(list(set(text)))
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
vocab_size = len(chars)
seq_len = 20

print(f"文本长度: {len(text)} 字符")
print(f"词汇表大小: {vocab_size}")
print(f"字符: {''.join(chars[:20])}...")

# 构造训练数据
X_data, y_data = [], []
for i in range(0, len(text) - seq_len):
    X_data.append([char2idx[c] for c in text[i:i+seq_len]])
    y_data.append([char2idx[c] for c in text[i+1:i+seq_len+1]])

X_tensor = torch.LongTensor(X_data)
y_tensor = torch.LongTensor(y_data)

### 2. 定义模型

运行 2. 定义模型 的代码，观察算法在该环节的行为。

In [ ]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
# --- 继续 ---
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        embeds = self.embedding(x)
        out, hidden = self.rnn(embeds, hidden)
        logits = self.fc(out)
        return logits, hidden

model = CharRNN(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

### 3. 训练

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
print("\n=== 训练 ===")
n_epochs = 100
for epoch in range(n_epochs):
    model.train()
    output, _ = model(X_tensor)
# --- 继续 ---
    loss = criterion(output.reshape(-1, vocab_size), y_tensor.reshape(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f"  Epoch {epoch+1}: Loss={loss.item():.4f}")

### 4. 文本生成函数

运行 4. 文本生成函数 的代码，观察算法在该环节的行为。

In [ ]:
def generate_text(model, start_text, length=50, temperature=0.8):
    """从起始文本生成指定长度的新文本"""
    model.eval()
    # 将起始文本转为索引
    input_seq = torch.LongTensor([[char2idx.get(c, 0) for c in start_text]])
    hidden = None
    result = list(start_text)

    with torch.no_grad():
        for _ in range(length):
            logits, hidden = model(input_seq, hidden)
            # temperature: 控制随机性（越低越确定，越高越随机）
            probs = torch.softmax(logits[0, -1] / temperature, dim=0)
            idx = torch.multinomial(probs, 1).item()
            result.append(idx2char[idx])
            input_seq = torch.LongTensor([[idx]])

    return "".join(result)

### 5. 生成文本示例

运行 5. 生成文本示例 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 生成文本（不同 temperature）===")
start = "人工智能"
for temp in [0.5, 0.8, 1.0, 1.2]:
    generated = generate_text(model, start, length=40, temperature=temp)
    print(f"  T={temp}: {generated}")

### 6. 不同 temperature 的效果

运行 6. 不同 temperature 的效果 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== Temperature 参数 ===")
print("T < 1.0: 更保守，选择概率高的词，文本更连贯但单调")
print("T = 1.0: 按原始概率采样")
print("T > 1.0: 更随机，文本更多样但可能不通顺")

### 7. 词级生成（简化示例）

运行 7. 词级生成（简化示例） 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 词级生成思路 ===")
print("1. 分词 → 构建词表（限制大小，如 top-5000）")
print("2. 训练 RNN/LSTM/Transformer 预测下一个词")
print("3. 用采样策略（贪心/Top-k/Top-p/Beam Search）生成")
print()
# --- 输出结果 ---
print("采样策略:")
print("  贪心: 始终选概率最高的词（确定性，缺乏多样性）")
print("  Top-k: 只从前 k 个候选中采样")
print("  Top-p (Nucleus): 从累积概率达到 p 的最小候选集中采样")
print("  Beam Search: 保留多个最优序列，适合翻译等任务")

print("\n=== 文本生成要点 ===")
print("- 自回归生成: 每次预测下一个 token，逐步生成")
print("- GPT 系列是当前最强的文本生成模型")
print("- Temperature 是控制生成质量/多样性的关键参数")
print("- 长文本生成需要处理重复和一致性问题")
# --- 输出结果 ---
print("- 评估困难: BLEU/ROUGE 只衡量表面相似度")

## 关键代码解释

贪心解码、Top-k 采样、Top-p（核）采样是三种常见的生成策略。

## 注意事项

1. 生成质量与多样性需要平衡（temperature 参数）
2. 重复生成是常见问题
3. 评估困难（BLEU、ROUGE 等自动指标不够完美）

## 总结与延伸

以上代码展示了 **文本生成** 的完整流程。

**解读要点**：
- 观察 **词汇表大小** 和 **向量维度** 是否合理
- 检查相似词查询结果是否符合语义直觉
- 关注分类任务中的 **TF-IDF 权重** 分布

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **GPT 系列**：自回归语言模型的巅峰
- **解码策略**：Beam Search、采样、对比搜索
- **可控生成**：如何让模型按指定风格/主题生成
